In [1]:
import numpy
import sys
import os
import matplotlib.pyplot as plt
# import skimage
import nrrd
from scipy import ndimage
import meshio
from skimage.morphology import skeletonize
from scipy.interpolate import splprep, splev
from sklearn.decomposition import PCA
from scipy.spatial import cKDTree
from scipy.interpolate import UnivariateSpline
import trimesh
from scipy.ndimage import median_filter

from scipy.spatial import cKDTree
from scipy.interpolate import UnivariateSpline
from skimage import measure
import tomllib

import pyvista as pv

from scipy.ndimage import uniform_filter1d

sys.path.insert(0, "../src")  # add Folder_2 path to search list

from meshing_functions import getSurfaceMesh, export_centerline,  tetra_mesh_from_stl, set_the_offset, tetra_shell_from_two_surfaces
from image_processing_functions import load_vti


with open("/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/T1_Extended_muscles/config.toml", mode="rb") as f:
    config = tomllib.load(f)

sequence = "T1"

if sequence == "T2":
    base            = config["data"]["base"]
    input_file      = base + config["data"]["nii_file_T2"]
    output_folder   = config["data"]["output_folder"]
    segment_filename = config["data"]["segment_filename_T2"]
    segment_folder  = config["data"]["segment_folder_T2"]
    centerline_folder = config["data"]["centerline_folder"]

else:
    base            = config["data"]["base"]
    input_file      = base + config["data"]["nii_file"]
    output_folder   = config["data"]["output_folder"]
    segment_filename = config["data"]["segment_filename"]
    segment_folder  = config["data"]["segment_folder"]
    centerline_folder = config["data"]["centerline_folder"]

export_name = base + output_folder + segment_filename[:-8]


In [2]:
n_theta_points = 40
smoothing_factor = 5

## Load image parameters

In [3]:
data, header = nrrd.read(base + segment_folder + segment_filename)

dirs = [numpy.array(d, dtype=float) for d in header["space directions"] if d is not None and not numpy.any(numpy.isnan(numpy.array(d, dtype=float)))]
dirs = numpy.array(dirs)

pixel_spacing = numpy.linalg.norm(dirs, axis=1)           # mm per voxel, in array axis order
direction = (dirs / pixel_spacing[:, None]).T          # columns = unit axis directions
origin = numpy.array(header["space origin"], dtype=float)

print("original pixel spacing = ", pixel_spacing)
print("segmentation origin = ", origin)

original pixel spacing =  [0.3125     0.3125     0.50000006]
segmentation origin =  [-55.95619965 -90.20069885  20.49090004]


## Load segments


In [4]:
segments = []
i = 0
while f"Segment{i}_Name" in header:
    name  = header[f"Segment{i}_Name"]
    label = int(header.get(f"Segment{i}_LabelValue", i + 1))
    layer = int(header.get(f"Segment{i}_Layer", 0))
    vol   = data[layer] if data.ndim == 4 else data
    segments.append((name, vol == label))
    print(f"{name}: layer={layer}, value={label}, voxels={segments[-1][1].sum()}")
    i += 1

combined = numpy.zeros(segments[0][1].shape, dtype=numpy.uint8)
for new_id, (name, mask) in enumerate(segments, start=1):
    combined[mask] = new_id          # later segments overwrite earlier ones where they overlap

data = combined
id_max = int(data.max())


optical nerve: layer=0, value=1, voxels=15624
eyeball: layer=0, value=3, voxels=160561
lateral rectus: layer=0, value=4, voxels=14436
medial rectus: layer=0, value=5, voxels=13164
inferior rectus: layer=0, value=6, voxels=11961
superior rectus: layer=0, value=7, voxels=13568
lens: layer=0, value=2, voxels=1508


## Resample

In [5]:
pixel_spacing = numpy.asarray(pixel_spacing, dtype=float)
target = pixel_spacing.min()

if not numpy.allclose(pixel_spacing, target):
    factors = pixel_spacing / target
    data = ndimage.zoom(data, factors, order=0)
    print("resampling factors = ", factors)
    print("new dimensions = ", data.shape)

pixel_spacing = target

print("Final uniform pixel_spacing = ", pixel_spacing)

resampling factors =  [1.         1.         1.60000019]
new dimensions =  (145, 209, 162)
Final uniform pixel_spacing =  0.3124999999999999


## Define IDs

In [6]:
eyeball_id = 2
lens_id = 7

muscle_ids = [3,4,1]
ON_id = 1



## Load masks: muslces, ON, eyeball

In [7]:
muslce_masks = []

for id in muscle_ids: 
    mask = numpy.zeros(data.shape, dtype=numpy.uint8)
    mask[(data>id-0.5) & (data<id+0.5 )]=1
    muslce_masks.append(mask)

ON_mask = numpy.zeros(data.shape, dtype=numpy.uint8)
ON_mask[(data>ON_id-0.5) & (data<ON_id+0.5 )]=1

eyeball_mask = numpy.zeros(data.shape, dtype=numpy.uint8)
eyeball_mask[(data>eyeball_id-0.5) & (data<eyeball_id+0.5 )]=1


# Part II

In [8]:
def get_local_frame(direction):
    """Calculates two vectors perpendicular to the spine direction."""
    # Pick a non-parallel vector to start the cross product
    # vec = numpy.array([1, 0, 0]) if abs(direction[0]) < 0.95 else numpy.array([0, 1, 0])

    vec_1 = numpy.array([1, 1, 0])
    vec = vec_1/numpy.linalg.norm(vec_1)

    v1 = numpy.cross(direction, vec)
    v1 /= numpy.linalg.norm(v1)
    v2 = numpy.cross(direction, v1)
    return v1, v2

def parallel_transport_frames(centerline):
    """Rotation-minimizing frames along a polyline. Returns tangents, v1, v2 (each N x 3)."""
    t = numpy.gradient(centerline, axis=0)
    t /= numpy.linalg.norm(t, axis=1, keepdims=True)

    ref = numpy.array([1.0, 0.0, 0.0]) if abs(t[0, 0]) < 0.9 else numpy.array([0.0, 1.0, 0.0])
    v1 = numpy.zeros_like(t)
    v1[0] = numpy.cross(t[0], ref)
    v1[0] /= numpy.linalg.norm(v1[0])

    for i in range(1, len(t)):
        v = v1[i - 1] - (v1[i - 1] @ t[i]) * t[i]       # drop component along new tangent
        v1[i] = v / numpy.linalg.norm(v)

    v2 = numpy.cross(t, v1)
    return t, v1, v2

In [9]:
from skimage import measure

def muscle_surface(mask, smooth_iter=20):
    verts, faces, _, _ = measure.marching_cubes(mask.astype(numpy.float32), level=0.5)
    surf = trimesh.Trimesh(vertices=verts, faces=faces, process=True)
    trimesh.smoothing.filter_taubin(surf, iterations=smooth_iter)   # smooth without shrinking
    return surf                                                     # in voxel coordinates

def radial_profiles_from_surface(centerline, surf, v1, v2, n_theta=40):
    theta = numpy.linspace(-numpy.pi, numpy.pi, n_theta, endpoint=False)
    n = len(centerline)

    dirs = (numpy.cos(theta)[None, :, None] * v1[:, None, :]
            + numpy.sin(theta)[None, :, None] * v2[:, None, :]).reshape(-1, 3)
    origins = numpy.repeat(centerline, n_theta, axis=0)

    radii = numpy.full(n * n_theta, numpy.nan)
    inside = surf.contains(centerline)                              # rays only from nodes inside the muscle
    use = numpy.repeat(inside, n_theta)

    locs, ray_idx, _ = surf.ray.intersects_location(origins[use], dirs[use], multiple_hits=False)
    ray_ids = numpy.flatnonzero(use)[ray_idx]
    radii[ray_ids] = numpy.linalg.norm(locs - origins[ray_ids], axis=1)
    return radii.reshape(n, n_theta)

In [10]:
def clamp_radii_by_curvature(centerline, radii, v1, v2, safety=0.8):
    t = numpy.gradient(centerline, axis=0)
    ds = numpy.linalg.norm(t, axis=1)
    t /= ds[:, None]
    dt = numpy.gradient(t, axis=0) / ds[:, None]
    kappa = numpy.linalg.norm(dt, axis=1)                      # curvature (1/voxel)
    normal = dt / numpy.maximum(kappa, 1e-12)[:, None]         # points to the inside of the bend

    n_theta = radii.shape[1]
    theta = numpy.linspace(-numpy.pi, numpy.pi, n_theta, endpoint=False)
    out = radii.copy()
    for i in range(len(centerline)):
        if kappa[i] < 1e-9:
            continue
        ray = numpy.cos(theta)[:, None] * v1[i] + numpy.sin(theta)[:, None] * v2[i]
        toward_inside = numpy.maximum(ray @ normal[i], 1e-6)
        out[i] = numpy.minimum(out[i], safety / (kappa[i] * toward_inside))
    return out

## Extract profiles

In [11]:
def extract_radial_profiles_from_mask(centerline, mask, pixel_spacing, n_theta=64):
    """
    Step 1: Casts fixed radiating angular rays outward from each centerline node 
    to measure the exact distance (radius) to the mask boundary.
    """
    voxel_coords = numpy.argwhere(mask)
    if len(voxel_coords) == 0:
        return numpy.full((len(centerline), n_theta), numpy.nan)
        
    tree = cKDTree(voxel_coords)
    n_nodes = len(centerline)
    
    # Grid to hold our raw radii: rows = slices, columns = fixed angles
    radii_grid = numpy.zeros((n_nodes, n_theta))
    
    # Generate the fixed, matching angular target steps once
    theta_steps = numpy.linspace(-numpy.pi, numpy.pi, n_theta, endpoint=False)
    
    for idx in range(n_nodes):
        p = centerline[idx]
        
        # Calculate local tracking frame
        direction = centerline[min(idx+1, n_nodes-1)] - centerline[max(idx-1, 0)]
        norm_dir = numpy.linalg.norm(direction)
        if norm_dir == 0:
            radii_grid[idx, :] = numpy.nan
            continue
        direction /= norm_dir
        v1, v2 = get_local_frame(direction)
        
        # Broad neighborhood check
        close_idx = tree.query_ball_point(p, r=60)
        if len(close_idx) < 10:
            radii_grid[idx, :] = numpy.nan # Outside the mask volume
            continue
            
        local_voxels = voxel_coords[close_idx] - p
        dist_plane = numpy.dot(local_voxels, direction)
        slab = numpy.abs(dist_plane) < 1.0
        local_voxels = local_voxels[slab]
        
        if len(local_voxels) < 10:
            radii_grid[idx, :] = numpy.nan
            continue
            
        # Project local voxels onto 2D plane
        x = local_voxels @ v1
        y = local_voxels @ v2
        
        # Calculate polar coordinates of the raw slice voxels
        vox_theta = numpy.arctan2(y, x)
        vox_r = numpy.sqrt(x**2 + y**2)
        
        # Map the voxel distances onto our fixed angular target steps
        for t_idx, target_theta in enumerate(theta_steps):
            # Find boundary voxels near this specific ray angle (e.g., within 6 degrees)
            angle_diff = numpy.abs(numpy.arctan2(numpy.sin(vox_theta - target_theta), numpy.cos(vox_theta - target_theta)))
            angular_slice_mask = angle_diff < (2*numpy.pi / n_theta)
            
            if numpy.any(angular_slice_mask):
                # The maximum radius along this ray represents the outer boundary wall
                radii_grid[idx, t_idx] = numpy.max(vox_r[angular_slice_mask])
            else:
                radii_grid[idx, t_idx] = numpy.nan
                
    return radii_grid


def smooth_and_extrapolate_radii(radii_grid, n_output_extended, smoothing=15.0):
    """
    PASS 1: Longitudinal 1st-order (linear) spline extrapolation down each angle.
    PASS 2: In-slice cyclic wrapping filters to polish the 2D cross-sections.
    """
    n_trimmed, n_theta = radii_grid.shape
    
    # Initialize the output grid to fit the full extended target length
    smooth_profiles = numpy.zeros((n_output_extended, n_theta))

    # =========================================================================
    # PASS 1: LONGITUDINAL SMOOTHING & LINEAR EXTRAPOLATION (Across-Slice First)
    # =========================================================================
    u_real = numpy.arange(n_trimmed)
    u_evaluation = numpy.arange(n_output_extended)
    
    for t_idx in range(n_theta):
        column = radii_grid[:, t_idx]
        valid_mask = ~numpy.isnan(column)
        
        # Repair internal gaps within this angular column before fitting
        if numpy.any(valid_mask) and numpy.sum(valid_mask) >= 3:
            # Linear spline (k=1) to project raw trajectory straight into extension zone
            rib_spline = UnivariateSpline(
                u_real[valid_mask], 
                column[valid_mask], 
                k=1, 
                s=smoothing, 
                ext=0
            )
            extrapolated_radii = rib_spline(u_evaluation)
            smooth_profiles[:, t_idx] = numpy.clip(extrapolated_radii, 0.1, None)
        else:
            smooth_profiles[:, t_idx] = 1.0

    # Apply a quick longitudinal filter to stabilize the tracking strings
    for t_idx in range(n_theta):
        smooth_profiles[:, t_idx] = uniform_filter1d(
            smooth_profiles[:, t_idx], 
            size=5, 
            mode='nearest'
        )

    # =========================================================================
    # PASS 2: IN-SLICE PERIMETER SMOOTHING (In-Slice Second)
    # =========================================================================
    # Now that the full grid (up to n_output_extended) is populated, we loop
    # through EVERY slice and polish its circular contour perimeter.
    for idx in range(n_output_extended):
        row_series = smooth_profiles[idx, :].copy()
        
        # Failsafe check for empty rows
        if numpy.all(numpy.isnan(row_series)):
            continue
            
        # Clean up any leftover interpolation gaps around the loop
        nans = numpy.isnan(row_series)
        if numpy.any(nans):
            xp = numpy.where(~nans)[0]
            fp = row_series[~nans]
            row_series[nans] = numpy.interp(numpy.where(nans)[0], xp, fp, period=n_theta)
            
        # Cyclic wrapping filters erase any remaining jagged voxel steps
        filtered_row = median_filter(row_series, size=5, mode='wrap')
        smooth_profiles[idx, :] = uniform_filter1d(filtered_row, size=5, mode='wrap')
        
    return smooth_profiles



## Generate mesh

In [12]:
def generate_radial_mesh(centerline, radii_grid, pixel_spacing_val, insertion_idx=None,
                         taper_start_len=12, taper_end_len=12, v1_frames=None, v2_frames=None):
    
    print("Centerline shape:", centerline.shape)
    print("Radii grid shape:", radii_grid.shape)

    n_sections, n_theta = radii_grid.shape

    if insertion_idx is None:
        insertion_idx = n_sections

    n_sections = min(n_sections, insertion_idx)
    vertices = []
    
    theta_steps = numpy.linspace(-numpy.pi, numpy.pi, n_theta, endpoint=False)
    
    start_point = centerline[0]
    end_point = centerline[n_sections - 1]
    
    v1_prev = None
    # THE CRITICAL ADDITION: Accumulate the total twist angle continuously
    cumulative_twist = 0.0

    # -------------------------------------------------------------------------
    # 1. Slice Ring Geometry Mapping with Cumulative Realignment
    # -------------------------------------------------------------------------
    for i in range(n_sections):
        p_node = centerline[i]

        if v1_frames is not None:
            v1, v2 = v1_frames[i], v2_frames[i]
        else:
            direction = centerline[min(i + 1, n_sections - 1)] - centerline[max(i - 1, 0)]
            direction /= numpy.linalg.norm(direction)
            v1, v2 = get_local_frame(direction)

        slice_theta = theta_steps
        r_slice = radii_grid[i, :].copy()
        

        # Store the current frame to guide the next slice transition
        v1_prev = v1
        
        r_slice = radii_grid[i, :].copy()
        
        # =========================================================================
        # PROXIMAL SMOOTH DOME (Muscle Origin / Start)
        # =========================================================================
        if i < taper_start_len:
            t = i / max(taper_start_len - 1, 1)   
            scale = numpy.sqrt(t * (2.0 - t))     
            
            p_node = (1.0 - t) * start_point + t * p_node
            r_slice = r_slice * scale
            
        # =========================================================================
        # DISTAL SMOOTH DOME (Eye Wall Insertion / End)
        # =========================================================================
        if i >= n_sections - taper_end_len:
            t = (i - (n_sections - taper_end_len)) / max(taper_end_len - 1, 1) 
            scale = numpy.sqrt((1.0 - t) * (2.0 - (1.0 - t)))
            
            p_node = (1.0 - t) * p_node + t * end_point
            r_slice = r_slice * scale
            
        r_slice = numpy.maximum(r_slice, 0.05)
        
        # Build the ring using the adjusted slice_theta steps
        ring = (
            p_node[None, :]
            + r_slice[:, None] * numpy.cos(slice_theta)[:, None] * v1[None, :]
            + r_slice[:, None] * numpy.sin(slice_theta)[:, None] * v2[None, :]
        )
        vertices.append(ring)
        
    vertices = numpy.vstack(vertices)
    
    # -------------------------------------------------------------------------
    # 2. Append Explicit Singular Tip Vertices for Cap Enclosures
    # -------------------------------------------------------------------------
    start_tip_idx = len(vertices)
    vertices = numpy.vstack([vertices, start_point[None, :]])
    
    end_tip_idx = len(vertices)
    vertices = numpy.vstack([vertices, end_point[None, :]])
    
    # -------------------------------------------------------------------------
    # 3. Surface Topology Mapping (Closed Cap Enclosure)
    # -------------------------------------------------------------------------
    faces = []
    
    # A. Close Proximal Cap (Start Point)
    for j in range(n_theta):
        j2 = (j + 1) % n_theta
        faces.append([start_tip_idx, j2, j])
        
    # B. Construct Continuous Central Tube Quads
    for i in range(n_sections - 1):
        base0 = i * n_theta
        base1 = (i + 1) * n_theta
        
        for j in range(n_theta):
            j2 = (j + 1) % n_theta
            
            v00 = base0 + j
            v01 = base0 + j2
            v10 = base1 + j
            v11 = base1 + j2
            
            faces.append([v00, v10, v11])
            faces.append([v00, v11, v01])
            
    # C. Close Distal Cap (End Point)
    final_ring_base = (n_sections - 1) * n_theta
    for j in range(n_theta):
        j2 = (j + 1) % n_theta
        faces.append([final_ring_base + j, end_tip_idx, final_ring_base + j2])
            
    # -------------------------------------------------------------------------
    # 4. Compile and Export Clean Simulation Mesh Asset
    # -------------------------------------------------------------------------
    mesh = trimesh.Trimesh(
        vertices=vertices * pixel_spacing_val,
        faces=numpy.asarray(faces),
        process=False
    )
    
    return mesh

## Find intersection of the extended centerline and the eyeball


In [13]:
def find_eye_intersection_index(centerline_extended, mask_eye):
    """
    Returns the index along the extended centerline that intersects the eye.
    """
    for idx, point in enumerate(centerline_extended):
        z, y, x = numpy.round(point).astype(int)
        
        # Check bounds
        if 0 <= z < mask_eye.shape[0] and 0 <= y < mask_eye.shape[1] and 0 <= x < mask_eye.shape[2]:
            if mask_eye[z, y, x] == 1:
                return idx  
                
    return len(centerline_extended) - 1


## Meshin loop init

In [14]:
def resample_centerline(centerline, step=1.0, smooth=0.25):
    """Smooth with a spline and resample at equal arc length (step in voxels)."""
    d = numpy.linalg.norm(numpy.diff(centerline, axis=0), axis=1)
    cl = centerline[numpy.concatenate([[True], d > 1e-6])]       # drop duplicate points

    tck, _ = splprep(cl.T, s=smooth * len(cl))                    # s ~ mean squared deviation per point
    u_dense = numpy.linspace(0, 1, 2000)
    pts = numpy.array(splev(u_dense, tck)).T
    arc = numpy.concatenate([[0], numpy.cumsum(numpy.linalg.norm(numpy.diff(pts, axis=0), axis=1))])

    n = max(int(arc[-1] / step) + 1, 4)
    u_new = numpy.interp(numpy.linspace(0, arc[-1], n), arc, u_dense)
    return numpy.array(splev(u_new, tck)).T

In [ ]:
n_theta_points = 40
smoothing_factor = 5
RESAMPLE_STEP = 1.0          # centerline node spacing in voxels

for id in range(len(muscle_ids)):

    muscle_id = muscle_ids[id]
    print(f"--- muscle {muscle_id} ---")

    initial_centerline = numpy.load(base + centerline_folder + "centerline_init_id_" + str(muscle_id) + ".npy")

    # 1. Smooth and resample at equal arc length
    centerline = resample_centerline(initial_centerline, step=RESAMPLE_STEP)

    # 2. Rotation-minimizing frames
    tangents, v1, v2 = parallel_transport_frames(centerline)

    # 3. Ray casting against the smoothed muscle surface
    surf = muscle_surface(muslce_masks[id])
    radii_raw = radial_profiles_from_surface(centerline, surf, v1, v2, n_theta=n_theta_points)

    # 4. Keep only the part of the centerline that lies inside the muscle
    valid = ~numpy.all(numpy.isnan(radii_raw), axis=1)
    if valid.sum() < 4:
        print("Too few nodes inside the muscle, skipping")
        continue
    first, last = numpy.flatnonzero(valid)[[0, -1]]
    centerline = centerline[first:last + 1]
    v1, v2 = v1[first:last + 1], v2[first:last + 1]
    radii_raw = radii_raw[first:last + 1]
    print(f"nodes used: {len(centerline)} (trimmed {first} at start, {len(valid) - 1 - last} at end)")

    # 5. Smooth along the muscle and around each ring (fills isolated gaps only)
    radii_processed = smooth_and_extrapolate_radii(
        radii_grid=radii_raw,
        n_output_extended=len(centerline),
        smoothing=smoothing_factor,
    )

    # 6. Prevent ring self-intersections on the inside of bends
    radii_processed = clamp_radii_by_curvature(centerline, radii_processed, v1, v2)

    # 7. Build the tube with the same frames
    mesh = generate_radial_mesh(
        centerline=centerline,
        radii_grid=radii_processed,
        pixel_spacing_val=pixel_spacing,
        insertion_idx=None,
        taper_start_len=15,
        taper_end_len=0,
        v1_frames=v1,
        v2_frames=v2,
    )

    mesh.export(export_name + "_RADIAL_id_" + str(muscle_id) + ".stl")

--- muscle 3 ---
nodes used: 103 (trimmed 2 at start, 0 at end)
Centerline shape: (103, 3)
Radii grid shape: (103, 40)
--- muscle 4 ---
nodes used: 90 (trimmed 2 at start, 0 at end)
Centerline shape: (90, 3)
Radii grid shape: (90, 40)
--- muscle 1 ---
nodes used: 101 (trimmed 15 at start, 0 at end)
Centerline shape: (101, 3)
Radii grid shape: (101, 40)
